# Padel AI Coach – analyse a new match on a Colab GPU
A full match is ~45–55k frames. On the laptop CPU, pose + ball inference would take many hours; on a T4 it takes ~30–60 min.
This notebook produces two files that the local app imports (**Matches → Import Colab file**):
* `<video>.npz` – players, 17 keypoints and ball for every frame,
* `<video>.json` – broadcast scoreboard readings (authoritative point winners), when a scoreboard is visible.

Needs: `MyDrive/padel/padel_bundle.zip`, the trained weights `MyDrive/padel/runs/{ball,pose}/weights/best.pt` (from `padel_train_models.ipynb`) and the match video in `MyDrive/padel/videos/`.
If the camera differs from the WPT broadcast view, first calibrate the court in the local app, rebuild the bundle (`python -m padelcoach export-training`) and re-upload it.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/padel'
!mkdir -p /content/padel /content/out
!unzip -oq $DRIVE/padel_bundle.zip -d /content/padel
%cd /content/padel
!pip -q install ultralytics easyocr

In [ ]:
VIDEO = f'{DRIVE}/videos/my_match.mp4'   # <- change to your video file name
POSE = f'{DRIVE}/runs/pose/weights/best.pt'
BALL = f'{DRIVE}/runs/ball/weights/best.pt'
!python colab_scripts/gpu_inference.py --video "$VIDEO" --pose "$POSE" --ball "$BALL" --out /content/out --scoreboard

In [ ]:
!ls -la /content/out && cp /content/out/* $DRIVE/
from google.colab import files
import glob
for f in glob.glob('/content/out/*'):
    files.download(f)

Put the video in `c:\Padel\workspaceideos\` (or upload it in the app), then on its card use **Import Colab file** for the `.npz` and the `.json`, choose source **import** and press **Analyse**.